In [1]:
import os
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split

In [2]:
# ==========================================
# 1. CONFIGURATION
# ==========================================
DATA_DIR = './processed_data' # Ensure this matches where you ran make_dataset.py
TRAIN_FILE = os.path.join(DATA_DIR, 'quickdraw_train.npz')
TEST_FILE = os.path.join(DATA_DIR, 'quickdraw_test.npz')

BATCH_SIZE = 128
EPOCHS = 5
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
print(f"Using device: {DEVICE}")


Using device: cpu


In [3]:
# ==========================================
# 2. DATASET CLASS (The NPZ Loader)
# ==========================================

class QuickDrawDataset(Dataset):
    def __init__(self, file_path, mode='train'):
        """
        Args:
            file_path (str): Path to the .npz file
            mode (str): 'train' (loads images & labels) or 'test' (loads images only)
        """
        self.mode = mode

        if not os.path.exists(file_path):
            raise FileNotFoundError(f"Could not find file: {file_path}")

        print(f"Loading {mode} data from {file_path}...")
        data = np.load(file_path)

        if mode == 'train':
            # Load x_train and y_train
            self.x = data['x_train']
            self.y = data['y_train']
            self.classes = data['class_names']
            print(f"Loaded {len(self.x)} training samples. Classes: {len(self.classes)}")

        elif mode == 'test':
            # Load test_images (for leaderboard inference)
            self.x = data['test_images']
            self.y = None
            print(f"Loaded {len(self.x)} test images.")

        # Pre-processing:
        # Convert to Float Tensor and Normalize (0-255 -> 0-1)
        self.x = torch.from_numpy(self.x).float() / 255.0

        if self.y is not None:
            self.y = torch.from_numpy(self.y).long()

    def __len__(self):
        return len(self.x)

    def __getitem__(self, idx):
        img = self.x[idx]
        if self.mode == 'train':
            label = self.y[idx]
            return img, label
        else:
            return img

In [4]:
# ==========================================
# 3. PREPARE DATALOADERS
# ==========================================

CLASSES = ['apple', 'baseballbat', 'basketball', 'clock', 'compass', 'cookie', 'donut', 'ladder', 'mountain', 'pizza', 'rabbit', 'soccerball', 'spider', 't-shirt', 'wheel']

# 1. Load the Training Data
full_train_dataset = QuickDrawDataset(TRAIN_FILE, mode='train')
NUM_CLASSES = len(full_train_dataset.classes)

# 2. Create Validation Split (80% Train / 20% Val)
train_size = int(0.8 * len(full_train_dataset))
val_size = len(full_train_dataset) - train_size
train_dataset, val_dataset = random_split(full_train_dataset, [train_size, val_size])

# 3. Create Loaders
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f"Train samples: {len(train_dataset)} | Validation samples: {len(val_dataset)}")

Loading train data from ./processed_data\quickdraw_train.npz...
Loaded 60000 training samples. Classes: 15
Train samples: 48000 | Validation samples: 12000


In [5]:
# ==========================================
# 4. YOUR IMPLEMENTATION HERE
# ==========================================
# Part A: The Pancake (Width Focus) — shallow but wide MLP

INPUT_SIZE = 28 * 28  # 784 flattened pixels
MAX_PARAMS = 3_000_000

class PancakeMLP(nn.Module):
    """Shallow, wide MLP: 1 or 2 hidden layers with many neurons (e.g. 1024, 2048)."""
    def __init__(self, input_dim=INPUT_SIZE, num_classes=NUM_CLASSES, hidden_width=1024, num_hidden_layers=2):
        super().__init__()
        self.input_dim = input_dim
        layers = []
        in_f = input_dim
        for _ in range(num_hidden_layers):
            layers.extend([nn.Linear(in_f, hidden_width), nn.ReLU(inplace=True)])
            in_f = hidden_width
        self.backbone = nn.Sequential(*layers)
        self.fc_out = nn.Linear(hidden_width, num_classes)

    def forward(self, x):
        # Flatten 28x28 -> 784 if needed
        x = x.view(x.size(0), -1)
        x = self.backbone(x)
        return self.fc_out(x)

def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

# Part A config: wide and shallow (2 hidden layers, 1024 units each; under 3M params)
model = PancakeMLP(hidden_width=1024, num_hidden_layers=2)
assert count_parameters(model) <= MAX_PARAMS, "Model exceeds 3M parameters!"

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

def train_epoch(model, loader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    correct, total = 0, 0
    for batch in loader:
        X, y = batch[0].to(device), batch[1].to(device)
        optimizer.zero_grad()
        logits = model(X)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
        _, pred = logits.max(1)
        total += y.size(0)
        correct += pred.eq(y).sum().item()
    return running_loss / len(loader), correct / total

def evaluate(model, loader, device):
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for batch in loader:
            X, y = batch[0].to(device), batch[1].to(device)
            logits = model(X)
            _, pred = logits.max(1)
            total += y.size(0)
            correct += pred.eq(y).sum().item()
    return correct / total

# Training loop (max 40 epochs per assignment)
EPOCHS_PANCAKE = 40
history = {"train_acc": [], "val_acc": [], "train_loss": []}

print("Part A: The Pancake (Width Focus)")
print(f"  Hidden: 2 layers x 1024 | Params: {count_parameters(model):,}")
print(f"  Epochs: {EPOCHS_PANCAKE}\n")

for epoch in range(EPOCHS_PANCAKE):
    model = model.to(DEVICE)
    train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, DEVICE)
    val_acc = evaluate(model, val_loader, DEVICE)
    history["train_acc"].append(train_acc)
    history["val_acc"].append(val_acc)
    history["train_loss"].append(train_loss)
    if (epoch + 1) % 5 == 0 or epoch == 0:
        print(f"Epoch {epoch+1:3d} | Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | Val Acc: {val_acc:.4f}")

print("\n--- Part A Summary ---")
print(f"Final Training Accuracy:   {history['train_acc'][-1]:.4f}")
print(f"Final Validation Accuracy: {history['val_acc'][-1]:.4f}")



Part A: The Pancake (Width Focus)
  Hidden: 2 layers x 1024 | Params: 1,868,815
  Epochs: 40

Epoch   1 | Train Loss: 1.1168 | Train Acc: 0.6325 | Val Acc: 0.7014
Epoch   5 | Train Loss: 0.4377 | Train Acc: 0.8492 | Val Acc: 0.7602
Epoch  10 | Train Loss: 0.1348 | Train Acc: 0.9531 | Val Acc: 0.7575
Epoch  15 | Train Loss: 0.0685 | Train Acc: 0.9762 | Val Acc: 0.7508
Epoch  20 | Train Loss: 0.0586 | Train Acc: 0.9805 | Val Acc: 0.7561
Epoch  25 | Train Loss: 0.0525 | Train Acc: 0.9827 | Val Acc: 0.7531
Epoch  30 | Train Loss: 0.0320 | Train Acc: 0.9901 | Val Acc: 0.7582
Epoch  35 | Train Loss: 0.0279 | Train Acc: 0.9916 | Val Acc: 0.7534
Epoch  40 | Train Loss: 0.0298 | Train Acc: 0.9909 | Val Acc: 0.7548

--- Part A Summary ---
Final Training Accuracy:   0.9909
Final Validation Accuracy: 0.7548


In [8]:
# ==========================================
# Part B: The Tower (Depth Focus) — deep but narrow MLP
# ==========================================
# Hypothesis: Can a hierarchical structure learn better features with fewer total parameters?
# Uses Batch Normalization and careful init to mitigate vanishing gradients (no skip connections).

class TowerMLP(nn.Module):
    """Deep, narrow MLP: 5+ hidden layers, few units per layer (e.g. 128, 256)."""
    def __init__(self, input_dim=INPUT_SIZE, num_classes=NUM_CLASSES, hidden_width=256, num_hidden_layers=6):
        super().__init__()
        layers = []
        in_f = input_dim
        for i in range(num_hidden_layers):
            layers.append(nn.Linear(in_f, hidden_width))
            layers.append(nn.BatchNorm1d(hidden_width))
            layers.append(nn.ReLU(inplace=True))
            in_f = hidden_width
        self.backbone = nn.Sequential(*layers)
        self.fc_out = nn.Linear(hidden_width, num_classes)
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
            elif isinstance(m, nn.BatchNorm1d):
                nn.init.ones_(m.weight)
                nn.init.zeros_(m.bias)

    def forward(self, x):
        x = x.view(x.size(0), -1)
        x = self.backbone(x)
        return self.fc_out(x)

tower_model = TowerMLP(hidden_width=256, num_hidden_layers=6)
assert count_parameters(tower_model) <= MAX_PARAMS, "Model exceeds 3M parameters!"

criterion_b = nn.CrossEntropyLoss()
optimizer_b = optim.Adam(tower_model.parameters(), lr=1e-3)

EPOCHS_TOWER = 40
history_b = {"train_acc": [], "val_acc": [], "train_loss": []}

print("Part B: The Tower (Depth Focus)")
print(f"  Hidden: 6 layers x 256 (with BatchNorm) | Params: {count_parameters(tower_model):,}")
print(f"  Epochs: {EPOCHS_TOWER}\n")

for epoch in range(EPOCHS_TOWER):
    tower_model = tower_model.to(DEVICE)
    train_loss, train_acc = train_epoch(tower_model, train_loader, criterion_b, optimizer_b, DEVICE)
    val_acc = evaluate(tower_model, val_loader, DEVICE)
    history_b["train_acc"].append(train_acc)
    history_b["val_acc"].append(val_acc)
    history_b["train_loss"].append(train_loss)
    if (epoch + 1) % 5 == 0 or epoch == 0:
        print(f"Epoch {epoch+1:3d} | Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | Val Acc: {val_acc:.4f}")

print("\n--- Part B Summary ---")
print(f"Final Training Accuracy:   {history_b['train_acc'][-1]:.4f}")
print(f"Final Validation Accuracy: {history_b['val_acc'][-1]:.4f}")

# Use Tower for submission (run inference cell below); to submit Pancake instead, set model = model before that.
model = tower_model

Part B: The Tower (Depth Focus)
  Hidden: 6 layers x 256 (with BatchNorm) | Params: 536,847
  Epochs: 40

Epoch   1 | Train Loss: 1.5891 | Train Acc: 0.5545 | Val Acc: 0.6686
Epoch   5 | Train Loss: 0.5736 | Train Acc: 0.8026 | Val Acc: 0.7236
Epoch  10 | Train Loss: 0.3188 | Train Acc: 0.8884 | Val Acc: 0.7212
Epoch  15 | Train Loss: 0.1882 | Train Acc: 0.9338 | Val Acc: 0.7288
Epoch  20 | Train Loss: 0.1288 | Train Acc: 0.9549 | Val Acc: 0.7292
Epoch  25 | Train Loss: 0.1117 | Train Acc: 0.9624 | Val Acc: 0.7305
Epoch  30 | Train Loss: 0.0928 | Train Acc: 0.9689 | Val Acc: 0.7325
Epoch  35 | Train Loss: 0.0759 | Train Acc: 0.9735 | Val Acc: 0.7302
Epoch  40 | Train Loss: 0.0680 | Train Acc: 0.9768 | Val Acc: 0.7373

--- Part B Summary ---
Final Training Accuracy:   0.9768
Final Validation Accuracy: 0.7373


In [16]:
# ==========================================
# Part C: The Champion — best architecture for leaderboard
# ==========================================
# Tuned from Part A/B: balanced depth+width, GELU, BatchNorm, Dropout, Weight Decay.
# Saves best model by validation accuracy for submission and GitHub.

BEST_WEIGHTS_PATH = "best_champion.pt"

class ChampionMLP(nn.Module):
    """Best MLP: moderate depth & width, BatchNorm, GELU, Dropout for generalization."""
    def __init__(self, input_dim=INPUT_SIZE, num_classes=NUM_CLASSES, hidden_width=512, num_hidden_layers=4,
                 dropout=0.35):
        super().__init__()
        layers = []
        in_f = input_dim
        for _ in range(num_hidden_layers):
            layers.append(nn.Linear(in_f, hidden_width))
            layers.append(nn.BatchNorm1d(hidden_width))
            layers.append(nn.GELU())
            layers.append(nn.Dropout(dropout))
            in_f = hidden_width
        self.backbone = nn.Sequential(*layers)
        self.fc_out = nn.Linear(hidden_width, num_classes)
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
            elif isinstance(m, nn.BatchNorm1d):
                nn.init.ones_(m.weight)
                nn.init.zeros_(m.bias)

    def forward(self, x):
        x = x.view(x.size(0), -1)
        x = self.backbone(x)
        return self.fc_out(x)

# Improved Champion: 5 layers x 512 (more capacity), dropout 0.38, cosine LR, input noise
champion_model = ChampionMLP(hidden_width=512, num_hidden_layers=5, dropout=0.38)
assert count_parameters(champion_model) <= MAX_PARAMS, "Model exceeds 3M parameters!"

criterion_c = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer_c = optim.AdamW(champion_model.parameters(), lr=2e-3, weight_decay=5e-4)
EPOCHS_CHAMPION = 40
scheduler_c = optim.lr_scheduler.CosineAnnealingLR(optimizer_c, T_max=EPOCHS_CHAMPION, eta_min=5e-5)
INPUT_NOISE_STD = 0.03  # Gaussian noise on input during training for better generalization

def get_val_loss(model, loader, criterion, device):
    model.eval()
    total_loss, n = 0.0, 0
    with torch.no_grad():
        for batch in loader:
            X, y = batch[0].to(device), batch[1].to(device)
            logits = model(X)
            total_loss += criterion(logits, y).item() * y.size(0)
            n += y.size(0)
    return total_loss / n

def train_epoch_with_noise(model, loader, criterion, optimizer, device, input_noise_std=0.0):
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    for batch in loader:
        X, y = batch[0].to(device), batch[1].to(device)
        if input_noise_std > 0:
            X = X + input_noise_std * torch.randn_like(X, device=device)
        optimizer.zero_grad()
        logits = model(X)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
        _, pred = logits.max(1)
        total += y.size(0)
        correct += pred.eq(y).sum().item()
    return running_loss / len(loader), correct / total

history_c = {"train_acc": [], "val_acc": [], "train_loss": [], "val_loss": []}
best_val_acc = 0.0
best_epoch = 0

print("Part C: The Champion (Leaderboard) [improved v2]")
print(f"  Hidden: 5 x 512 | BatchNorm, GELU, Dropout=0.38 | Label smoothing=0.1 | CosineAnnealingLR | Input noise={INPUT_NOISE_STD}")
print(f"  Params: {count_parameters(champion_model):,} | Epochs: {EPOCHS_CHAMPION}\n")

for epoch in range(EPOCHS_CHAMPION):
    champion_model = champion_model.to(DEVICE)
    train_loss, train_acc = train_epoch_with_noise(champion_model, train_loader, criterion_c, optimizer_c, DEVICE, INPUT_NOISE_STD)
    val_acc = evaluate(champion_model, val_loader, DEVICE)
    val_loss = get_val_loss(champion_model, val_loader, criterion_c, DEVICE)
    scheduler_c.step()
    history_c["train_acc"].append(train_acc)
    history_c["val_acc"].append(val_acc)
    history_c["train_loss"].append(train_loss)
    history_c["val_loss"].append(val_loss)

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_epoch = epoch + 1
        torch.save(champion_model.state_dict(), BEST_WEIGHTS_PATH)

    if (epoch + 1) % 5 == 0 or epoch == 0:
        print(f"Epoch {epoch+1:3d} | Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | Val Acc: {val_acc:.4f} | Val Loss: {val_loss:.4f}")

# Load best weights for submission (weights_only=True avoids pickle security warning)
state_dict = torch.load(BEST_WEIGHTS_PATH, map_location=DEVICE, weights_only=True)
champion_model.load_state_dict(state_dict)
model = champion_model

print("\n--- Part C Summary ---")
print(f"Best Validation Accuracy: {best_val_acc:.4f} (saved at epoch {best_epoch})")
print(f"Best weights saved to: {BEST_WEIGHTS_PATH}")
print(f"Final Training Accuracy: {history_c['train_acc'][-1]:.4f}")

Part C: The Champion (Leaderboard) [improved v2]
  Hidden: 5 x 512 | BatchNorm, GELU, Dropout=0.38 | Label smoothing=0.1 | CosineAnnealingLR | Input noise=0.03
  Params: 1,465,359 | Epochs: 40

Epoch   1 | Train Loss: 3.4980 | Train Acc: 0.4021 | Val Acc: 0.6201 | Val Loss: 1.5874
Epoch   5 | Train Loss: 1.4516 | Train Acc: 0.6724 | Val Acc: 0.7273 | Val Loss: 1.2669
Epoch  10 | Train Loss: 1.1812 | Train Acc: 0.7526 | Val Acc: 0.7669 | Val Loss: 1.1401
Epoch  15 | Train Loss: 1.0446 | Train Acc: 0.8089 | Val Acc: 0.7863 | Val Loss: 1.0900
Epoch  20 | Train Loss: 0.9405 | Train Acc: 0.8515 | Val Acc: 0.7958 | Val Loss: 1.0824
Epoch  25 | Train Loss: 0.8587 | Train Acc: 0.8858 | Val Acc: 0.7989 | Val Loss: 1.0872
Epoch  30 | Train Loss: 0.8054 | Train Acc: 0.9077 | Val Acc: 0.7984 | Val Loss: 1.1051
Epoch  35 | Train Loss: 0.7726 | Train Acc: 0.9194 | Val Acc: 0.7987 | Val Loss: 1.1054
Epoch  40 | Train Loss: 0.7607 | Train Acc: 0.9252 | Val Acc: 0.8009 | Val Loss: 1.1162

--- Part C Su

In [17]:
import pandas as pd
from sklearn.metrics import accuracy_score

# ==========================================
# 5. INFERENCE & LEADERBOARD VERIFICATION
# ==========================================
# Uses the Champion model with best saved weights (from Part C).
# For standalone submission from GitHub: python inference.py (requires best_champion.pt)
print("\n" + "="*40)
print("   GENERATING SUBMISSION FILE")
print("="*40)
# 1. Load Test Images
test_dataset = QuickDrawDataset(TEST_FILE, mode='test')
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

def get_predictions(model, loader):
    model.eval()
    model.to(DEVICE)
    preds = []
    with torch.no_grad():
        for batch in loader:
            X = batch.to(DEVICE)
            outputs = model(X)
            _, predicted = torch.max(outputs, 1)
            preds.extend(predicted.cpu().numpy())
    return preds

# 2. Run Inference
print("Running inference on test set...")
predictions = get_predictions(model, test_loader)

# 3. Save as Comma-Separated Text File
submission_file = "submission.txt"
print(f"Saving predictions to '{submission_file}'...")

# Convert list of ints to comma-separated string (e.g., "0,4,9,2...")
submission_string = ",".join(map(str, predictions))

with open(submission_file, "w") as f:
    f.write(submission_string)
print(f"-> Copy & paste the results of this file to the portal.")


   GENERATING SUBMISSION FILE
Loading test data from ./processed_data\quickdraw_test.npz...
Loaded 15000 test images.
Running inference on test set...
Saving predictions to 'submission.txt'...
-> Copy & paste the results of this file to the portal.


In [15]:
def print_model_size(model):
    total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"\nModel Statistics:")
    print(f"  Total Parameters: {total_params:,}")
    if total_params > 3000000:
        print("  ⚠️ WARNING: You are over the 3M parameter limit!")
    else:
        print("  ✅ Parameter count is within limits.")

print_model_size(model)


Model Statistics:
  Total Parameters: 1,201,679
  ✅ Parameter count is within limits.
